# ML Evaluator - o mesmo fluxo, agora em código

Este notebook refaz, passo a passo, exatamente o que a ferramenta Streamlit faz por baixo dos panos.
O objetivo é mostrar como um experimento de aprendizado supervisionado é conduzido em ambiente de
**estudo**, antes de virar uma aplicação.

As cinco etapas são as mesmas da ferramenta:

1. Carga dos dados
2. Seleção das features (X) e do target (y)
3. Seleção do tipo de problema (classificação ou regressão)
4. Seleção do modelo
5. Tabela de métricas

> Para executar no Google Colab: `Arquivo > Abrir notebook > Upload` e envie este arquivo.


## Preparando o ambiente


In [ ]:
# No Colab estes pacotes já vêm instalados. A linha abaixo garante o ambiente
# em qualquer máquina; se já estiver tudo presente, ela apenas confirma.
!pip install -q scikit-learn pandas matplotlib seaborn


In [ ]:
# Importações. Cada bloco corresponde a uma responsabilidade do fluxo.
import pandas as pd
import numpy as np

# Dados de exemplo (embutidos no scikit-learn, não precisam de download)
from sklearn.datasets import load_breast_cancer, load_diabetes

# Separação treino/teste
from sklearn.model_selection import train_test_split

# Pré-processamento
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Modelos (a mesma lista oferecida pela ferramenta)
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.neighbors import KNeighborsClassifier

# Métricas
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

# Semente única: garante que repetir o experimento devolva o mesmo resultado.
# Sem isso você não consegue saber se a métrica mudou por causa da sua alteração
# ou por causa do acaso.
SEMENTE = 42
TAMANHO_TESTE = 0.2


# Passo a passo


## Passo 1: Carregar os dados

Na ferramenta este passo é o upload do arquivo. Aqui você tem duas opções:

- **Opção A (padrão):** usar um conjunto de exemplo que já vem no scikit-learn.
- **Opção B:** enviar o seu próprio CSV - descomente o bloco indicado.


In [ ]:
# ----- Opção A: conjunto de exemplo -------------------------------------
# Câncer de mama -> problema de CLASSIFICAÇÃO (o alvo é uma categoria)
# Diabetes       -> problema de REGRESSÃO     (o alvo é um número contínuo)

USAR_EXEMPLO = "classificacao"   # troque para "regressao" para ver o outro caminho

if USAR_EXEMPLO == "classificacao":
    bruto = load_breast_cancer(as_frame=True)
    dados = bruto.frame.copy()
    # Converte o código numérico do alvo em rótulo legível: é assim que o dado
    # costuma chegar na vida real, e força o fluxo a lidar com alvo textual.
    dados["diagnostico"] = bruto.target.map({0: "maligno", 1: "benigno"})
    dados = dados.drop(columns=["target"])
    COLUNA_ALVO = "diagnostico"
else:
    bruto = load_diabetes(as_frame=True)
    dados = bruto.frame.copy()
    COLUNA_ALVO = "target"

# ----- Opção B: seu próprio arquivo -------------------------------------
# from google.colab import files
# enviados = files.upload()
# nome = next(iter(enviados))
# dados = pd.read_csv(nome, sep=None, engine="python")   # sep=None detecta o separador
# COLUNA_ALVO = "nome_da_sua_coluna_alvo"

print(f"Linhas: {dados.shape[0]}  |  Colunas: {dados.shape[1]}")
print(f"Coluna alvo: {COLUNA_ALVO}")
dados.head()


In [ ]:
# Olhar os dados ANTES de modelar é parte do método, não perda de tempo.
# Esta tabela é a mesma da aba "Perfil das colunas" da ferramenta.
perfil = pd.DataFrame({
    "tipo": dados.dtypes.astype(str),
    "ausentes": dados.isna().sum(),
    "ausentes_%": (100 * dados.isna().mean()).round(2),
    "distintos": dados.nunique(),
})
perfil.head(15)


## Passo 2: Selecionar as features (X) e o target (y)

`y` é a resposta que queremos prever. `X` são as pistas que o modelo pode usar.

**Atenção ao vazamento de alvo:** se uma coluna de X já contém a resposta (o parecer final,
a data da decisão, um código atribuído depois dela), a métrica vai para perto de 100% e o
modelo fracassa na vida real. A ferramenta desmarca essas colunas automaticamente; aqui a
responsabilidade é sua.


In [ ]:
y = dados[COLUNA_ALVO]
X = dados.drop(columns=[COLUNA_ALVO])

# Descarte de colunas inúteis: constantes não informam nada, e colunas de texto
# com valor distinto em quase toda linha costumam ser identificadores.
constantes = [c for c in X.columns if X[c].nunique(dropna=True) <= 1]
identificadores = [
    c for c in X.columns
    if not pd.api.types.is_numeric_dtype(X[c]) and X[c].nunique(dropna=True) > 50
]
descartadas = constantes + identificadores
if descartadas:
    print("Colunas descartadas:", descartadas)
    X = X.drop(columns=descartadas)

# Checagem simples de vazamento: correlação quase perfeita com um alvo numérico.
if pd.api.types.is_numeric_dtype(y):
    correlacoes = X.select_dtypes("number").corrwith(y).abs()
    suspeitas = correlacoes[correlacoes > 0.999].index.tolist()
    if suspeitas:
        print("SUSPEITA DE VAZAMENTO (confira antes de usar):", suspeitas)

print(f"X tem {X.shape[1]} coluna(s) e {X.shape[0]} linha(s).")
X.head()


## Passo 3: Definir o tipo de problema

**Classificação** prevê uma categoria (aprovado/reprovado, benigno/maligno).
**Regressão** prevê um número em escala contínua (preço, temperatura).

A regra prática: alvo textual, ou numérico com pouquíssimos valores distintos, é classificação.


In [ ]:
def inferir_tipo_de_problema(alvo: pd.Series) -> str:
    """Sugere o tipo de problema a partir do conteúdo da coluna alvo."""
    limpo = alvo.dropna()
    if not pd.api.types.is_numeric_dtype(limpo):
        return "classificacao"
    distintos = limpo.nunique()
    if distintos <= 2:
        return "classificacao"
    inteiro = pd.api.types.is_integer_dtype(limpo)
    if inteiro and distintos <= 20 and distintos / len(limpo) <= 0.05:
        return "classificacao"
    return "regressao"


TIPO_PROBLEMA = inferir_tipo_de_problema(y)
print("Tipo de problema:", TIPO_PROBLEMA)
print("Primeiros valores do alvo:", list(pd.Series(y).unique()[:10]))


## Passo 4: Escolher e treinar o modelo

Esta é a mesma lista de modelos oferecida pela ferramenta. Cada família carrega uma suposição
diferente sobre os dados - não existe um melhor modelo universal.

| Modelo | O que faz | Quando usar |
| --- | --- | --- |
| Regressão Logística | Estima a probabilidade de cada categoria | Classificação binária em que interpretar o resultado importa |
| Random Forest | Combina centenas de árvores por votação ou média | Ponto de partida mais seguro, sem ajuste fino |
| Árvore de Decisão | Cria regras encadeadas do tipo se/então | Quando explicar o raciocínio é essencial |
| K Vizinhos (KNN) | Olha os registros históricos mais parecidos | Classificação simples, com poucas colunas |
| Regressão Linear | Ajusta a reta que melhor se encaixa nos dados | Alvo numérico com relação aproximadamente proporcional |


In [ ]:
MODELOS = {
    "classificacao": {
        "Regressão Logística": LogisticRegression(max_iter=1000, random_state=SEMENTE),
        "Random Forest": RandomForestClassifier(n_estimators=200, random_state=SEMENTE),
        "Árvore de Decisão": DecisionTreeClassifier(random_state=SEMENTE),
        "K Vizinhos (KNN)": KNeighborsClassifier(n_neighbors=5),
    },
    "regressao": {
        "Regressão Linear": LinearRegression(),
        "Random Forest": RandomForestRegressor(n_estimators=200, random_state=SEMENTE),
        "Árvore de Decisão": DecisionTreeRegressor(random_state=SEMENTE),
    },
}

# Troque aqui para experimentar outro modelo e comparar as métricas.
MODELO_ESCOLHIDO = "Random Forest"

estimador = MODELOS[TIPO_PROBLEMA][MODELO_ESCOLHIDO]
print("Modelo selecionado:", MODELO_ESCOLHIDO)
estimador


In [ ]:
# Pré-processamento. Ele precisa estar DENTRO do pipeline: assim a mediana usada
# para preencher vazios e a escala usada para padronizar são aprendidas apenas
# no conjunto de treino. Aprender isso na tabela inteira é vazamento de dados -
# o erro mais comum e mais difícil de perceber.
colunas_numericas = X.select_dtypes(include="number").columns.tolist()
colunas_categoricas = [c for c in X.columns if c not in colunas_numericas]

transformadores = []
if colunas_numericas:
    transformadores.append(("numericas", Pipeline([
        ("imputacao", SimpleImputer(strategy="median")),
        ("padronizacao", StandardScaler()),
    ]), colunas_numericas))
if colunas_categoricas:
    transformadores.append(("categoricas", Pipeline([
        ("imputacao", SimpleImputer(strategy="most_frequent")),
        ("codificacao", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]), colunas_categoricas))

preprocessamento = ColumnTransformer(transformadores)

modelo = Pipeline([
    ("preprocessamento", preprocessamento),
    ("modelo", estimador),
])

# Separação treino/teste. Em classificação estratificamos, para preservar a
# proporção das categorias nos dois lados.
estratificar = y if TIPO_PROBLEMA == "classificacao" else None
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TAMANHO_TESTE, random_state=SEMENTE, stratify=estratificar
)

print("Treino:", X_train.shape[0], "registros  |  Teste:", X_test.shape[0], "registros")

modelo.fit(X_train, y_train)
y_pred = modelo.predict(X_test)
print("Modelo treinado.")


## Passo 5: Tabela de métricas

As métricas são sempre calculadas no conjunto de **teste** - os registros que o modelo nunca viu.
É essa separação que impede um resultado otimista demais.

Em classificação, a acurácia sozinha engana: ela é uma média e pode esconder desempenho ruim na
categoria menos frequente. Por isso olhamos precisão, revocação e F1 **por classe**.


In [ ]:
if TIPO_PROBLEMA == "classificacao":
    print(f"Acurácia geral: {accuracy_score(y_test, y_pred):.4f}")

    relatorio = classification_report(y_test, y_pred, output_dict=True, zero_division=0)
    tabela_metricas = pd.DataFrame([
        {
            "Categoria": classe,
            "Precisão": valores["precision"],
            "Revocação": valores["recall"],
            "F1": valores["f1-score"],
            "Casos no teste": int(valores["support"]),
        }
        for classe, valores in relatorio.items()
        if classe not in ("accuracy", "macro avg", "weighted avg")
    ]).round(4)
else:
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    tabela_metricas = pd.DataFrame([
        {"Métrica": "R²", "Valor": r2_score(y_test, y_pred),
         "O que significa": "Fração da variação do alvo que o modelo explica; 1 é perfeito."},
        {"Métrica": "MAE", "Valor": mae,
         "O que significa": "Erro médio em módulo, na mesma unidade do alvo."},
        {"Métrica": "RMSE", "Valor": np.sqrt(mse),
         "O que significa": "Como o MAE, mas punindo erros grandes com mais peso."},
        {"Métrica": "MSE", "Valor": mse,
         "O que significa": "Média dos erros ao quadrado; base de cálculo do RMSE."},
    ]).round(4)

tabela_metricas


### Visualização do desempenho

Um número resume; um gráfico mostra *onde* o modelo erra. Em classificação, a matriz de confusão
revela qual categoria está sendo confundida com qual. Em regressão, o gráfico de resíduos revela
se o erro é uniforme ou cresce em alguma faixa de valores.


In [ ]:
if TIPO_PROBLEMA == "classificacao":
    rotulos = sorted(pd.Series(y_test).unique())
    matriz = confusion_matrix(y_test, y_pred, labels=rotulos)
    plt.figure(figsize=(5, 4))
    sns.heatmap(matriz, annot=True, fmt="d", cmap="Blues",
                xticklabels=rotulos, yticklabels=rotulos)
    plt.xlabel("Previsto pelo modelo")
    plt.ylabel("Valor real")
    plt.title("Matriz de confusão")
    plt.tight_layout()
    plt.show()
else:
    residuos = np.asarray(y_test, dtype=float) - np.asarray(y_pred, dtype=float)
    figura, eixos = plt.subplots(1, 2, figsize=(11, 4))
    eixos[0].scatter(y_pred, y_test, alpha=0.6)
    limite = [float(np.min(y_test)), float(np.max(y_test))]
    eixos[0].plot(limite, limite, "--", color="darkorange")
    eixos[0].set_xlabel("Previsto")
    eixos[0].set_ylabel("Real")
    eixos[0].set_title("Previsto x Real (a diagonal é o acerto perfeito)")
    eixos[1].scatter(y_pred, residuos, alpha=0.6)
    eixos[1].axhline(0, linestyle="--", color="darkorange")
    eixos[1].set_xlabel("Previsto")
    eixos[1].set_ylabel("Resíduo (real - previsto)")
    eixos[1].set_title("Resíduos")
    plt.tight_layout()
    plt.show()


## Extra: prever um caso novo, com medida de confiança

É o que a tela 'Avaliar novos casos' faz. O ponto importante: um modelo não devolve verdade,
devolve uma aposta - e a aposta precisa vir acompanhada de quanto ele confia nela.


In [ ]:
# Usamos o primeiro registro do teste como "caso novo" para o exemplo.
caso_novo = X_test.iloc[[0]]
previsao = modelo.predict(caso_novo)[0]

print("Previsão:", previsao)

if TIPO_PROBLEMA == "classificacao" and hasattr(modelo.named_steps["modelo"], "predict_proba"):
    probabilidades = modelo.predict_proba(caso_novo)[0]
    classes = modelo.named_steps["modelo"].classes_
    distribuicao = pd.Series(probabilidades, index=classes).sort_values(ascending=False)
    print(f"Confiança na categoria prevista: {distribuicao.iloc[0]:.1%}")
    print()
    print("Probabilidade por categoria:")
    print(distribuicao.round(4).to_string())
elif TIPO_PROBLEMA == "regressao":
    # Sem probabilidade, a incerteza vem da dispersão do erro no conjunto de teste.
    desvio = float(np.std(np.asarray(y_test, dtype=float) - np.asarray(y_pred, dtype=float), ddof=1))
    margem = 1.96 * desvio
    print(f"Com 95% de chance o valor real está entre "
          f"{previsao - margem:.2f} e {previsao + margem:.2f}")

print()
print("Valor real deste registro:", y_test.iloc[0])


## O que levar deste notebook

- O **teste nunca participa do treino**. Toda métrica confiável nasce dessa separação.
- **Pré-processamento vai dentro do pipeline**, senão o conjunto de teste contamina o treino.
- **Acurácia é uma média** e esconde desempenho ruim em categorias pouco frequentes.
- **Métrica perto de 100% é motivo de desconfiança**, não de comemoração: quase sempre é vazamento de alvo.
- **Semente fixa** é o que permite comparar dois experimentos e atribuir a diferença à sua mudança.

A ferramenta Streamlit deste projeto automatiza exatamente estes passos - e avisa quando alguma
dessas armadilhas aparece.
